In [2]:
#r "C:\Users\user\Desktop\Nurmanov\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

Console.WriteLine("=== Замеры производительности Round Robin планировщика ===\n");

var commandCounts = new int[] { 1, 2, 4, 8, 16, 32 };
int stepsPerCommand = 5;
int warmupRuns = 2;
int measureRuns = 5;

var sequentialTimes = new double[commandCounts.Length];
var roundRobinTimes = new double[commandCounts.Length];

for (int i = 0; i < commandCounts.Length; i++)
{
    int totalCommands = commandCounts[i];
    
    double seqTotal = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        var sw = Stopwatch.StartNew();
        for (int j = 0; j < totalCommands; j++)
        {
            var cmd = new WorkCommand(stepsPerCommand);
            while (!cmd.IsCompleted)
                cmd.Execute();
        }
        sw.Stop();
        if (run >= warmupRuns)
            seqTotal += sw.Elapsed.TotalMilliseconds;
    }
    sequentialTimes[i] = seqTotal / measureRuns;

    double rrTotal = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        var commands = new WorkCommand[totalCommands];
        for (int j = 0; j < totalCommands; j++)
            commands[j] = new WorkCommand(stepsPerCommand);
        
        var planner = new RoundRobinScheduler();
        var worker = new ServerThread(planner);
        
        foreach (var cmd in commands)
            planner.Add(cmd);
        
        var sw = Stopwatch.StartNew();
        worker.Start();
        
        while (!commands.All(c => c.IsCompleted))
            Thread.Sleep(1);
        
        sw.Stop();
        worker.HardStop();
        worker.Join();
        
        if (run >= warmupRuns)
            rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    roundRobinTimes[i] = rrTotal / measureRuns;
    
    double overhead = roundRobinTimes[i] - sequentialTimes[i];
    double overheadPercent = (overhead / sequentialTimes[i]) * 100.0;
    
    Console.WriteLine($"Команд: {totalCommands,3} | Последовательно: {sequentialTimes[i],8:F2} мс | Round Robin: {roundRobinTimes[i],8:F2} мс | Накладные: {overheadPercent,6:F1}%");
}

var plot1 = new Plot();
plot1.Title("Время выполнения: Последовательно vs Round Robin");
plot1.XLabel("Количество длительных команд");
plot1.YLabel("Время (мс)");

var xs = commandCounts.Select(c => (double)c).ToArray();

var seqScatter = plot1.Add.Scatter(xs, sequentialTimes);
seqScatter.LegendText = "Последовательное выполнение";
seqScatter.MarkerSize = 8;
seqScatter.LineWidth = 2;

var rrScatter = plot1.Add.Scatter(xs, roundRobinTimes);
rrScatter.LegendText = "Round Robin планировщик";
rrScatter.MarkerSize = 8;
rrScatter.LineWidth = 2;

plot1.ShowLegend();

var fn1 = "plot_scheduler_comparison.png";
plot1.SavePng(fn1, 800, 600);
display(HTML($"<img src='{fn1}?t={DateTime.Now.Ticks}' width='700'/>"));

var plot2 = new Plot();
plot2.Title("Накладные расходы Round Robin планировщика");
plot2.XLabel("Количество длительных команд");
plot2.YLabel("Накладные расходы (%)");

var overheads = new double[commandCounts.Length];
for (int i = 0; i < commandCounts.Length; i++)
{
    overheads[i] = ((roundRobinTimes[i] - sequentialTimes[i]) / sequentialTimes[i]) * 100.0;
}

var overheadBars = plot2.Add.Bars(xs, overheads);
overheadBars.LegendText = "Накладные расходы";

var line15 = plot2.Add.HorizontalLine(15.0);
line15.LineColor = ScottPlot.Color.FromHex("#FF0000");
line15.LineWidth = 2;
line15.LegendText = "Порог 15%";

plot2.ShowLegend();

var fn2 = "plot_scheduler_overhead.png";
plot2.SavePng(fn2, 800, 600);
display(HTML($"<img src='{fn2}?t={DateTime.Now.Ticks}' width='700'/>"));

var plot3 = new Plot();
plot3.Title("Ускорение Round Robin относительно последовательного выполнения");
plot3.XLabel("Количество длительных команд");
plot3.YLabel("Ускорение (раз)");

var speedups = new double[commandCounts.Length];
for (int i = 0; i < commandCounts.Length; i++)
{
    speedups[i] = sequentialTimes[i] / roundRobinTimes[i];
}

var speedupBars = plot3.Add.Bars(xs, speedups);
speedupBars.LegendText = "Ускорение";

var line1 = plot3.Add.HorizontalLine(1.0);
line1.LineColor = ScottPlot.Color.FromHex("#FF0000");
line1.LineWidth = 2;
line1.LegendText = "Паритет (1.0x)";

plot3.ShowLegend();

var fn3 = "plot_scheduler_speedup.png";
plot3.SavePng(fn3, 800, 600);
display(HTML($"<img src='{fn3}?t={DateTime.Now.Ticks}' width='700'/>"));

Console.WriteLine($"\nГрафики сохранены: {fn1}, {fn2}, {fn3}");

public class WorkCommand : ILongCommand
{
    private readonly int _maxSteps;
    private int _currentStep;
    
    public bool IsCompleted => _currentStep >= _maxSteps;
    
    public WorkCommand(int maxSteps)
    {
        _maxSteps = maxSteps;
    }
    
    public void Execute()
    {
        if (!IsCompleted)
        {
            double sum = 0;
            for (int i = 0; i < 50000; i++)
                sum += Math.Sin(i) * Math.Cos(i);
            
            _currentStep++;
        }
    }
}

Installed Packages ScottPlot, 5.0.54

=== Замеры производительности Round Robin планировщика ===

Команд:   1 | Последовательно:     5.94 мс | Round Robin:    16.33 мс | Накладные:  175.0%
Команд:   2 | Последовательно:    14.71 мс | Round Robin:    19.44 мс | Накладные:   32.2%
Команд:   4 | Последовательно:    23.19 мс | Round Robin:    31.25 мс | Накладные:   34.8%
Команд:   8 | Последовательно:    47.43 мс | Round Robin:    51.74 мс | Накладные:    9.1%
Команд:  16 | Последовательно:    95.01 мс | Round Robin:   103.12 мс | Накладные:    8.5%
Команд:  32 | Последовательно:   192.44 мс | Round Robin:   215.29 мс | Накладные:   11.9%



Графики сохранены: plot_scheduler_comparison.png, plot_scheduler_overhead.png, plot_scheduler_speedup.png
